In [22]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['Api_key']=os.getenv("HF_Token")
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [23]:
#Data Ingestion - from website we need to scrape the data

from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.langchain.com/langsmith/observability-quickstart")


In [ ]:
docs_loaded = loader.load()

# len(docs[0].page_content)

801

In [25]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap = 200)
docs = text_splitter.split_documents(docs_loaded)

In [26]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")


db = FAISS.from_documents(docs,embeddings)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2465.91it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
#Retieval chain
retriever = db.as_retriever()
result = retriever.invoke("such as an LLM call or a retrieval step.")
result[0].page_content

'LangSmith addresses this by providing end-to-end visibility into how your application handles a request. Each request generates a trace, which captures the full record of what happened. Within a trace are individual runs, the specific operations your application performed, such as an LLM call or a retrieval step. Tracing runs allows you to inspect, debug, and validate your application’s behavior.\nIn this quickstart, you will set up a minimal Retrieval Augmented Generation (RAG) application and add tracing with LangSmith. You will:'

In [29]:
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""
Answer the following question based only on the provided context:
<context>
{context}
</context>
"""
)

from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,
    max_retries=2
)

document_Chain = create_stuff_documents_chain(model,prompt)



In [30]:
document_Chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001BAB3C45490>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001BAB3C36810>, model_name='llama-3.1

In [32]:
from langchain_core.documents import Document

document_Chain.invoke({
    "input": "LangSmith addresses this by providing end-to-end visibility into how your application handles a request.",
    "context":[Document(page_content = "LangSmith addresses this by providing end-to-end visibility into how your application handles a request. Each request generates a trace, which captures the full record of what happened. Within a trace are individual runs, the specific operations your application performed, such as an LLM call or a retrieval step. Tracing runs allows you to inspect, debug, and validate your application’s behavior.")]
})

"LangSmith provides end-to-end visibility into how an application handles a request by generating a trace for each request. This trace captures the full record of what happened, including individual runs such as LLM calls or retrieval steps, allowing for inspection, debugging, and validation of the application's behavior."

In [36]:
retriever = db.as_retriever()

from langchain_classic.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever, document_Chain)

In [39]:
# lets get the reponse from LLM

response = retrieval_chain.invoke({"input":"LangSmith addresses this by providing end-to-end visibility into how your application handles a request."})

response

{'input': 'LangSmith addresses this by providing end-to-end visibility into how your application handles a request.',
 'context': [Document(id='28218159-1b45-4c6d-be6d-64e4bbe5d2c7', metadata={'source': 'https://docs.langchain.com/langsmith/observability-quickstart', 'title': 'Tracing quickstart - Docs by LangChain', 'language': 'en'}, page_content='LangSmith addresses this by providing end-to-end visibility into how your application handles a request. Each request generates a trace, which captures the full record of what happened. Within a trace are individual runs, the specific operations your application performed, such as an LLM call or a retrieval step. Tracing runs allows you to inspect, debug, and validate your application’s behavior.\nIn this quickstart, you will set up a minimal Retrieval Augmented Generation (RAG) application and add tracing with LangSmith. You will:'),
  Document(id='f3ca8232-b8eb-4be2-a533-268acf53dd9e', metadata={'source': 'https://docs.langchain.com/langs